In [0]:
%pip install geopandas

In [0]:
%sql
USE CATALOG `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`;
USE SCHEMA `hydraa`;

In [0]:
import geopandas as gpd
import os
from pyspark.sql.functions import col, lower, trim

# 1. Define Paths
BRONZE_FILE_PATH = "/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/bronze/hyderabad_lakes_raw.geojson"
SILVER_TABLE_NAME = "/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/silver_hydra_lakes"

print("🔍 Reading raw spatial data using GeoPandas...")
# Load the raw GeoJSON into a GeoDataFrame on the driver node
gdf = gpd.read_file(BRONZE_FILE_PATH)

# Inspect what columns exist in the dataset to clean them up dynamically
print(f"📊 Original columns found: {list(gdf.columns)}")

# 2. Geometry Transformation
# Convert the complex shapely geometric objects into a plain WKT string for Spark compatibility
print("📐 Transforming spatial geometries to WKT format...")
gdf['wkt_geometry'] = gdf['geometry'].apply(lambda x: x.wkt if x else None)

# Drop the native GeoPandas geometry object before converting to a Spark DataFrame
df_cleaned = gdf.drop(columns=['geometry'])

# 3. Port to Spark DataFrame
print("⚡ Converting to Apache Spark DataFrame...")
spark_df = spark.createDataFrame(df_cleaned)

# 4. Data Cleaning & Standardisation
# Standardise columns (lowercase names, drop null fields, trim whitespace)
for col_name in spark_df.columns:
    spark_df = spark_df.withColumnRenamed(col_name, col_name.lower().strip())

# Rename wkt_geometry back if it got modified, and filter out null shapes
spark_df = spark_df.filter(col("wkt_geometry").isNotNull())

# 5. Write to Delta Lake
print(f"💾 Saving processed spatial records to Delta Table: {SILVER_TABLE_NAME}...")
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_TABLE_NAME)

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_hydra_lakes")

print("✅ Silver Layer successfully created!")


In [0]:
#Read output from the above cell

spark.read.format("delta").load(SILVER_TABLE_NAME).display()